In [10]:
from pymongo import MongoClient
from neo4j import GraphDatabase

In [11]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "Niveau99"))
client = MongoClient("mongodb://localhost:27017/")
poetry_db = client["poetry"]

In [12]:
import pandas as pd

In [13]:
df = pd.DataFrame(poetry_db.books.find({}))
df.head()

,_id,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,69e8ac2044e2f3dc8aa255a6,,1,[],US,eng,"[{'count': '8', 'name': 'to-read'}, {'count': ...",,false,3.83,,[],Number 30 in a series of literary pamphlets pu...,Paperback,https://www.goodreads.com/book/show/16037549-v...,"[{'author_id': '15585', 'role': ''}]","Houghton, Mifflin and Company",80,1,,11,,1887,https://www.goodreads.com/book/show/16037549-v...,https://images.gr-assets.com/books/1348176637m...,16037549,3,5212748,Vision of Sir Launfal and Other Poems,Vision of Sir Launfal and Other Poems
1,69e8ac2044e2f3dc8aa255a7,0811223981,2,[],US,,"[{'count': '100', 'name': 'to-read'}, {'count'...",,false,3.83,B00U2WY9U8,[],Fairy Tales gathers the unconventional verse d...,Paperback,https://www.goodreads.com/book/show/22466716-f...,"[{'author_id': '16073', 'role': ''}, {'author_...",New Directions,128,20,9780811223980,4,,2015,https://www.goodreads.com/book/show/22466716-f...,https://images.gr-assets.com/books/1404958407m...,22466716,37,41905435,Fairy Tales: Dramolettes,Fairy Tales: Dramolettes
2,69e8ac2044e2f3dc8aa255a8,0374428115,7,[],US,,"[{'count': '32', 'name': 'to-read'}, {'count':...",,false,4.38,,[],Three poems describe the nighttime adventures ...,Paperback,https://www.goodreads.com/book/show/926662.Gro...,"[{'author_id': '18540', 'role': ''}, {'author_...",Farrar Straus Giroux,,12,9780374428112,7,,2008,https://www.goodreads.com/book/show/926662.Gro...,https://s.gr-assets.com/assets/nophoto/book/11...,926662,45,911665,Growltiger's Last Stand and Other Poems,Growltiger's Last Stand and Other Poems
3,69e8ac2044e2f3dc8aa255a9,0156182890,12,[],US,,"[{'count': '554', 'name': 'to-read'}, {'count'...",,false,3.71,B00IWTRB1W,"[1230072, 315167, 676169, 18522, 124335, 88263...",A modern verse play about the search for meani...,Paperback,https://www.goodreads.com/book/show/926667.The...,"[{'author_id': '18540', 'role': ''}]",Mariner Books,190,18,9780156182898,3,,1964,https://www.goodreads.com/book/show/926667.The...,https://images.gr-assets.com/books/1382939971m...,926667,115,995066,The Cocktail Party,The Cocktail Party
4,69e8ac2044e2f3dc8aa255aa,1942004192,4,[],US,eng,"[{'count': '228', 'name': 'to-read'}, {'count'...",,false,5.00,,"[25869488, 23630890, 25448131, 25464039, 42166...",Louder Than Everything You Love is about trans...,Paperback,https://www.goodreads.com/book/show/29065952-l...,"[{'author_id': '14308759', 'role': ''}]",ELJ Publications,118,23,9781942004196,12,First,2015,https://www.goodreads.com/book/show/29065952-l...,https://images.gr-assets.com/books/1455198396m...,29065952,9,49294781,Louder Than Everything You Love,Louder Than Everything You Love


In [19]:
def normalize_language(code):
    if pd.isna(code) or code in ("", "--"):
        return "UNK"
    code = code.lower().strip()

    # Modern English variants — collapse regional codes only
    if code in {"eng", "en", "en-us", "en-ca", "en-gb"}:
        return "eng"
    # enm (Middle English) and ang (Old English) kept separate

    # Modern German — collapse code variants only
    if code in {"ger", "deu"}:
        return "ger"
    # gmh (Middle High German) kept separate

    # Modern French — collapse code variants only
    if code in {"fre", "fra", "fr"}:
        return "fre"
    # fro (Old French) and frm (Middle French) kept separate

    # Modern Dutch — collapse code variants only
    if code in {"nl", "dut", "nld"}:
        return "dut"
    # dum (Middle Dutch) kept separate

    # Modern Greek — collapse code variants only
    if code in {"gre", "ell"}:
        return "gre"
    # grc (Ancient Greek) kept separate

    # Modern Persian — collapse code variants only
    if code in {"per", "fas", "pes"}:
        return "per"
    # peo (Old Persian) kept separate

    # Norwegian — Bokmål and Nynorsk are separate written standards but most
    # poetry catalogs treat them as one. Collapsing.
    if code in {"nor", "nob", "nno"}:
        return "nor"

    return code

In [29]:
DB_NAME = "goodreads-poetry"

In [28]:
def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

In [30]:
# LANG_MIN_COUNT = 50  # below this → OTHER bucket

# df["language_code_norm"] = df["language_code"].apply(normalize_language)

# counts = df["language_code_norm"].value_counts()
# keep = set(counts[counts >= LANG_MIN_COUNT].index)
# df["language_node"] = df["language_code_norm"].where(
#     df["language_code_norm"].isin(keep), "OTHER"
# )

# print("language node distribution:")
# print(df["language_node"].value_counts())

# # ----- 2. Create :Language nodes --------------------------------------------
# def create_language_index(tx):
#     tx.run("CREATE INDEX language_code_idx IF NOT EXISTS FOR (l:Language) ON (l.code)")

# def create_language_nodes(tx, codes):
#     tx.run("""
#         UNWIND $codes AS c
#         MERGE (:Language {code: c})
#     """, codes=codes)

# language_codes = sorted(df["language_node"].unique())
# print(f"\ncreating {len(language_codes)} language nodes")

# with driver.session(database=DB_NAME) as session:
#     session.execute_write(create_language_index)
#     session.execute_write(create_language_nodes, language_codes)

# # ----- 3. Create :IN_LANGUAGE edges -----------------------------------------
# def create_in_language_edges(tx, edges):
#     tx.run("""
#         UNWIND $edges AS e
#         MATCH (b:Book {book_id: e.book_id})
#         MATCH (l:Language {code: e.language})
#         MERGE (b)-[:IN_LANGUAGE]->(l)
#     """, edges=edges)

# in_language_edges = (
#     df[["book_id", "language_node"]]
#     .rename(columns={"language_node": "language"})
#     .assign(book_id=lambda x: x["book_id"].astype(str))
#     .to_dict("records")
# )

# with driver.session(database=DB_NAME) as session:
#     for chunk in chunked(in_language_edges, 10_000):
#         session.execute_write(create_in_language_edges, chunk)

# print(f"created {len(in_language_edges)} in_language edges")

language node distribution:
language_node
unk      19465
eng       9643
ara       1608
per        953
spa        837
OTHER      517
por        462
ita        455
tur        369
dut        229
ind        222
bul        205
fre        200
ger        193
gre        186
fin        169
cze        154
swe        116
rum        100
dan         92
nor         83
pol         72
mul         72
ben         57
rus         55
Name: count, dtype: int64

creating 25 language nodes
created 36514 in_language edges


In [37]:
def normalize_format(fmt):
    if pd.isna(fmt) or fmt == "":
        return "UNK"
    return fmt.strip().title()  # "paperback" and "PAPERBACK" both → "Paperback"

In [38]:
FORMAT_MIN_COUNT = 50  # tune after seeing the distribution

df["format_norm"] = df["format"].apply(normalize_format)

counts = df["format_norm"].value_counts()
print(counts)  # eyeball before bucketing

keep = set(counts[counts >= FORMAT_MIN_COUNT].index)
df["format_node"] = df["format_norm"].where(df["format_norm"].isin(keep), "OTHER")

print("\nfinal format node distribution:")
print(df["format_node"].value_counts())

format_norm
Paperback                             20147
UNK                                    7025
Hardcover                              6749
Ebook                                   904
Kindle Edition                          329
                                      ...  
Chap/E-Chap                               1
Brochura                                  1
Paperback / Shwmyz                        1
Hardcover Art And Poetry Gift Book        1
Relie                                     1
Name: count, Length: 169, dtype: int64

final format node distribution:
format_node
Paperback                20147
UNK                       7025
Hardcover                 6749
Ebook                      904
OTHER                      514
Kindle Edition             329
Unknown Binding            224
Mass Market Paperback      179
Audio Cd                   164
Chapbook                   148
Audiobook                   80
Leather Bound               51
Name: count, dtype: int64


In [39]:
# ============================================================================
# :Format nodes and :IN_FORMAT edges
# ============================================================================

# ----- 1. Create :Format nodes ----------------------------------------------
def create_format_index(tx):
    tx.run("CREATE INDEX format_name_idx IF NOT EXISTS FOR (f:Format) ON (f.name)")

def create_format_nodes(tx, names):
    tx.run("""
        UNWIND $names AS n
        MERGE (:Format {name: n})
    """, names=names)

format_names = sorted(df["format_node"].unique())
print(f"creating {len(format_names)} format nodes")

with driver.session(database=DB_NAME) as session:
    session.execute_write(create_format_index)
    session.execute_write(create_format_nodes, format_names)

# ----- 2. Create :IN_FORMAT edges -------------------------------------------
def create_in_format_edges(tx, edges):
    tx.run("""
        UNWIND $edges AS e
        MATCH (b:Book {book_id: e.book_id})
        MATCH (f:Format {name: e.format})
        MERGE (b)-[:IN_FORMAT]->(f)
    """, edges=edges)

in_format_edges = (
    df[["book_id", "format_node"]]
    .rename(columns={"format_node": "format"})
    .assign(book_id=lambda x: x["book_id"].astype(str))
    .to_dict("records")
)

with driver.session(database=DB_NAME) as session:
    for chunk in chunked(in_format_edges, 10_000):
        session.execute_write(create_in_format_edges, chunk)

print(f"created {len(in_format_edges)} in_format edges")




creating 12 format nodes
created 36514 in_format edges


In [40]:
def normalize_publisher(name):
    if pd.isna(name) or name == "":
        return "UNK"
    return name.strip()

PUBLISHER_MIN_COUNT = 50  # tune after seeing the distribution

df["publisher_norm"] = df["publisher"].apply(normalize_publisher)

counts = df["publisher_norm"].value_counts()
print(f"distinct publishers (after normalization): {len(counts)}")
print(counts.head(30))

keep = set(counts[counts >= PUBLISHER_MIN_COUNT].index)
df["publisher_node"] = df["publisher_norm"].where(
    df["publisher_norm"].isin(keep), "OTHER"
)

print("\nfinal publisher node distribution (top 30):")
print(df["publisher_node"].value_counts().head(30))
print(f"\ntotal publisher nodes: {df['publisher_node'].nunique()}")

distinct publishers (after normalization): 8186
publisher_norm
UNK                                            6033
W. W. Norton  Company                           320
Farrar, Straus and Giroux                       305
Penguin Classics                                302
Penguin Books                                   291
New Directions                                  261
Oxford University Press, USA                    240
Copper Canyon Press                             240
Knopf                                           227
Graywolf Press                                  184
W. W. Norton & Company                          180
Createspace Independent Publishing Platform     172
Dover Publications                              168
Faber  Faber                                    165
University of Pittsburgh Press                  158
Faber & Faber                                   146
Mariner Books                                   146
Oxford University Press                         146
E

In [41]:
# ----- 2. Create :Publisher nodes -------------------------------------------
def create_publisher_index(tx):
    tx.run("CREATE INDEX publisher_name_idx IF NOT EXISTS FOR (p:Publisher) ON (p.name)")

def create_publisher_nodes(tx, names):
    tx.run("""
        UNWIND $names AS n
        MERGE (:Publisher {name: n})
    """, names=names)

publisher_names = sorted(df["publisher_node"].unique())
print(f"\ncreating {len(publisher_names)} publisher nodes")

with driver.session(database=DB_NAME) as session:
    session.execute_write(create_publisher_index)
    session.execute_write(create_publisher_nodes, publisher_names)

# ----- 3. Create :PUBLISHED_BY edges ----------------------------------------
def create_published_by_edges(tx, edges):
    tx.run("""
        UNWIND $edges AS e
        MATCH (b:Book {book_id: e.book_id})
        MATCH (p:Publisher {name: e.publisher})
        MERGE (b)-[:PUBLISHED_BY]->(p)
    """, edges=edges)

published_by_edges = (
    df[["book_id", "publisher_node"]]
    .rename(columns={"publisher_node": "publisher"})
    .assign(book_id=lambda x: x["book_id"].astype(str))
    .to_dict("records")
)

with driver.session(database=DB_NAME) as session:
    for chunk in chunked(published_by_edges, 10_000):
        session.execute_write(create_published_by_edges, chunk)

print(f"created {len(published_by_edges)} published_by edges")


creating 86 publisher nodes
created 36514 published_by edges


In [ ]:
import json
from pathlib import Path

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [43]:
# ----- 1. Defensive type coercion -------------------------------------------

df["num_pages"] = pd.to_numeric(df["num_pages"], errors="coerce")
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce")
df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")
df["ratings_count"] = pd.to_numeric(df["ratings_count"], errors="coerce").fillna(0).astype(int)
df["text_reviews_count"] = pd.to_numeric(df["text_reviews_count"], errors="coerce").fillna(0).astype(int)

def to_ebook_flag(v):
    if pd.isna(v):
        return 0
    if isinstance(v, bool):
        return int(v)
    return 1 if str(v).strip().lower() == "true" else 0

df["is_ebook_flag"] = df["is_ebook"].apply(to_ebook_flag)

# Missingness flags — computed BEFORE imputation
df["is_num_pages_missing"] = df["num_pages"].isna().astype(int)
df["is_year_missing"] = df["publication_year"].isna().astype(int)
df["is_avg_rating_meaningful"] = (df["ratings_count"] > 0).astype(int)

In [44]:
# ----- 2. Normalization stats (Phase 1: full corpus) ------------------------

num_pages_median = float(df["num_pages"].median())
year_median = float(df["publication_year"].median())
year_mean = float(df["publication_year"].mean())
year_std = float(df["publication_year"].std())

norm_stats = {
    "num_pages_median": num_pages_median,
    "year_median": year_median,
    "year_mean": year_mean,
    "year_std": year_std,
    "_note": "Phase 1: stats computed over full corpus, not training split. Tighten in Phase 2.",
}

with open(OUT_DIR / "book_norm_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=2)